# Resolution-Aware CCTV Planner Presentation Notebook

This notebook presents the CCTV placement model at a high level. It provides a walkthrough of the floor-plan representation, model settings, methodology, and results across the traced floor plans.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.planner._shared.config import PlannerConfig
from src.planner.phase05_visualization import (
    plot_dori_map,
    plot_selected_configurations,
)
from src.planner.presentation import (
    DEFAULT_PRESENTATION_FLOORPLAN_NAMES,
    build_aligned_baseline_rows,
    build_candidate_summary_rows,
    build_floorplan_catalog_rows,
    build_settings_rows,
    compute_common_delta_k_range,
    plot_candidate_set_comparison,
    plot_metric_by_delta_k,
    plot_metric_by_k,
    plot_summary_table,
    require_floorplan_min_k,
    resolve_presentation_floorplan_results,
    select_showcase_k_values,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.labelsize"] = 10


## Load Presentation Data

The notebook loads the traced floor plans, summary data for candidate locations, and the result sets used for the budget comparisons shown below.

In [ ]:
CONFIG = PlannerConfig()
presentation_results = resolve_presentation_floorplan_results(
    floorplan_names=DEFAULT_PRESENTATION_FLOORPLAN_NAMES,
    base_config=CONFIG,
)
catalog_rows = build_floorplan_catalog_rows(presentation_results)
candidate_summary_rows = build_candidate_summary_rows(presentation_results)
settings_rows = build_settings_rows(CONFIG)
aligned_baseline_rows = build_aligned_baseline_rows(presentation_results)
common_delta_k_values = compute_common_delta_k_range(presentation_results)
showcase_k_values_by_floorplan = {
    result.floorplan.name: select_showcase_k_values(result, count=3)
    for result in presentation_results
}

{
    "floorplans": [result.floorplan.name for result in presentation_results],
    "configured_k_values": list(CONFIG.k_values),
    "common_delta_k_values": list(common_delta_k_values),
    "showcase_k_values_by_floorplan": showcase_k_values_by_floorplan,
}


## Floor-Plan Catalog

Each traced floor plan carries its own grid shape, cell counts, and presentation baseline `min_k`. The `min_k` value is **not** an optimization parameter; it is metadata used only to align cross-floorplan comparisons so that each layout begins from a more comparable starting line.

In [ ]:
figure, axis = plt.subplots(figsize=(12, 3.6))
plot_summary_table(
    catalog_rows,
    columns=[
        "name",
        "grid_shape",
        "open_cells",
        "solid_cells",
        "null_cells",
        "grid_cell_size_m",
        "min_k",
    ],
    column_labels=[
        "Floor plan",
        "Grid shape",
        "Open",
        "Solid",
        "Null",
        "Cell size (m)",
        "min K",
    ],
    ax=axis,
    title="Traced Floor-Plan Catalog",
)
plt.show()


## Model Representation

The model uses a **tri-state occupancy grid**:

- `-1` = null / out-of-scope cells outside the interior footprint
- `0` = open cells that are valid surveillance targets
- `1` = solid cells that represent walls or blockers

Candidate generation is also represented in two layers:

- the full **eligible** set of open cells adjacent to a non-open 4-neighbor
- the reduced **optimization-facing** candidate set used in the budget analysis

That distinction matters because the optimization is not run on every eligible boundary cell. Instead, it uses a thinner representative set along the walls and boundary structure.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 10))
for axis, result in zip(axes.flat, presentation_results):
    result.floorplan.plot(
        ax=axis,
        title=f"{result.floorplan.name} tri-state grid",
        show_colorbar=False,
    )
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(len(presentation_results), 2, figsize=(11, 4 * len(presentation_results)))
if len(presentation_results) == 1:
    axes = [axes]

for row_axes, result in zip(axes, presentation_results):
    plot_candidate_set_comparison(
        result.floorplan,
        result.phase01_artifacts,
        axes=row_axes,
        eligible_title=f"{result.floorplan.name} eligible candidates",
        final_title=f"{result.floorplan.name} optimization candidates",
    )
plt.tight_layout()
plt.show()

candidate_names = [str(row["name"]) for row in candidate_summary_rows]
eligible_candidate_counts = [int(row["eligible_candidates"]) for row in candidate_summary_rows]
optimization_candidate_counts = [int(row["optimization_candidates"]) for row in candidate_summary_rows]
candidate_reduction_pct = [float(row["candidate_reduction_pct"]) for row in candidate_summary_rows]
x = np.arange(len(candidate_names), dtype=float)
bar_width = 0.36

figure, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].bar(x - bar_width / 2, eligible_candidate_counts, width=bar_width, label="Eligible", color="#d96c4b")
axes[0].bar(x + bar_width / 2, optimization_candidate_counts, width=bar_width, label="Optimization", color="#2f9e44")
axes[0].set_xticks(x)
axes[0].set_xticklabels(candidate_names, rotation=0)
axes[0].set_ylabel("Candidate count")
axes[0].set_title("Candidate counts by floor plan")
axes[0].legend()
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(candidate_names, candidate_reduction_pct, color="#4f6ddf")
axes[1].set_ylabel("Reduction (%)")
axes[1].set_title("Candidate-set reduction after thinning")
axes[1].grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Model Settings

The analysis uses **45° orientation increments**, a camera field of view of `90°`, and a budget sweep of `K = 10..25`.

In [ ]:
figure, axis = plt.subplots(figsize=(12, 4.6))
plot_summary_table(
    settings_rows,
    columns=["setting", "value"],
    column_labels=["Setting", "Implemented value"],
    ax=axis,
    title="Model Settings Used in This Analysis",
)
plt.show()


## Methodology

The workflow follows five stages:

1. **Candidate generation** from tri-state floor plans
2. **Visibility analysis** using conservative grid line-of-sight
3. **Scoring** with field-of-view filtering and DORI quality levels
4. **Optimization** under a camera budget `K`
5. **Visualization** of the final spatial score surface and metrics

The optimization itself operates on a finite representation derived from the floor-plan grid. It does not reason directly over raw images or continuous geometry.

In [ ]:
pipeline_steps = [
    "01\nCandidate\ngeneration",
    "02\nVisibility",
    "03\nScoring",
    "04\nOptimization",
    "05\nVisualization",
]

figure, axis = plt.subplots(figsize=(14, 2.8))
axis.axis("off")
x_positions = np.linspace(0.08, 0.92, len(pipeline_steps))
for index, (x_position, step_label) in enumerate(zip(x_positions, pipeline_steps)):
    axis.text(
        x_position,
        0.5,
        step_label,
        ha="center",
        va="center",
        fontsize=11,
        bbox={"boxstyle": "round,pad=0.45", "facecolor": "#f4f4f4", "edgecolor": "#444444"},
    )
    if index < len(pipeline_steps) - 1:
        axis.annotate(
            "",
            xy=(x_positions[index + 1] - 0.09, 0.5),
            xytext=(x_position + 0.09, 0.5),
            arrowprops={"arrowstyle": "->", "linewidth": 1.6, "color": "#444444"},
        )
axis.set_title("Planner Pipeline Overview")
plt.show()


## Simulation Results by Floor Plan

The next figures keep **absolute camera budget `K`** on the horizontal axis. This view answers the within-floorplan question: how much DORI-weighted coverage improves as the allowed number of cameras increases.

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_metric_by_k(
    presentation_results,
    "total_dori_score",
    ax=axes[0, 0],
    title="Total DORI score by K",
    ylabel="Total DORI score",
)
plot_metric_by_k(
    presentation_results,
    "detection_plus_pct",
    ax=axes[0, 1],
    title="Detection+ coverage by K",
    ylabel="Coverage (%)",
)
plot_metric_by_k(
    presentation_results,
    "observation_plus_pct",
    ax=axes[0, 2],
    title="Observation+ coverage by K",
    ylabel="Coverage (%)",
)
plot_metric_by_k(
    presentation_results,
    "recognition_plus_pct",
    ax=axes[1, 0],
    title="Recognition+ coverage by K",
    ylabel="Coverage (%)",
)
plot_metric_by_k(
    presentation_results,
    "identification_pct",
    ax=axes[1, 1],
    title="Identification coverage by K",
    ylabel="Coverage (%)",
)
plot_metric_by_k(
    presentation_results,
    "blind_spot_pct",
    ax=axes[1, 2],
    title="Blind-spot percentage by K",
    ylabel="Open-cell blind spots (%)",
)
plt.tight_layout()
plt.show()


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for result in presentation_results:
    floorplan_name = result.floorplan.name
    showcase_k_values = showcase_k_values_by_floorplan[floorplan_name]
    resolved_artifacts = sorted(
        result.visualization_artifacts_by_k,
        key=lambda artifacts: int(artifacts.solved_k),
    )
    k_values = [int(artifacts.solved_k) for artifacts in resolved_artifacts]
    total_dori_values = [float(artifacts.metrics.total_dori_score) for artifacts in resolved_artifacts]
    blind_spot_values = [float(artifacts.metrics.blind_spot_pct) for artifacts in resolved_artifacts]
    axes[0].plot(k_values, total_dori_values, linewidth=1.8, label=floorplan_name)
    axes[1].plot(k_values, blind_spot_values, linewidth=1.8, label=floorplan_name)

    highlight_total_x = []
    highlight_total_y = []
    highlight_blind_y = []
    for k_value in showcase_k_values:
        artifacts = result.get_visualization_artifacts(k_value)
        highlight_total_x.append(k_value)
        highlight_total_y.append(float(artifacts.metrics.total_dori_score))
        highlight_blind_y.append(float(artifacts.metrics.blind_spot_pct))

    axes[0].scatter(highlight_total_x, highlight_total_y, s=55, zorder=4)
    axes[1].scatter(highlight_total_x, highlight_blind_y, s=55, zorder=4)

axes[0].set_title("Showcase checkpoints on total DORI curves")
axes[0].set_xlabel("Absolute camera budget K")
axes[0].set_ylabel("Total DORI score")
axes[0].grid(True, alpha=0.25)
axes[0].legend()

axes[1].set_title("Showcase checkpoints on blind-spot curves")
axes[1].set_xlabel("Absolute camera budget K")
axes[1].set_ylabel("Open-cell blind spots (%)")
axes[1].grid(True, alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()


## Cross-Floorplan Comparison Aligned by `min_k`

The next comparison changes the x-axis to **`delta_k = K - min_k`**. This keeps the comparison fairer across layouts with different room counts and different baseline camera needs.

For the floor plans used here, the shared aligned range is derived from the available budget sweep rather than fixed in advance.

In [ ]:
{
    "common_delta_k_values": list(common_delta_k_values),
    "baseline_actual_k_by_floorplan": {
        result.floorplan.name: require_floorplan_min_k(result.floorplan)
        for result in presentation_results
    },
}


In [ ]:
figure, axis = plt.subplots(figsize=(14, 3.8))
plot_summary_table(
    aligned_baseline_rows,
    columns=[
        "name",
        "min_k",
        "k",
        "total_dori_score",
        "detection_plus_pct",
        "observation_plus_pct",
        "recognition_plus_pct",
        "identification_pct",
        "blind_spot_pct",
    ],
    column_labels=[
        "Floor plan",
        "min K",
        "Actual K",
        "Total DORI",
        "Detection+ %",
        "Observation+ %",
        "Recognition+ %",
        "Identification %",
        "Blind spot %",
    ],
    ax=axis,
    title="Aligned Baseline Comparison at delta_k = 0",
)
plt.show()


In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_metric_by_delta_k(
    presentation_results,
    "total_dori_score",
    ax=axes[0, 0],
    title="Total DORI score by delta_k",
    ylabel="Total DORI score",
)
plot_metric_by_delta_k(
    presentation_results,
    "detection_plus_pct",
    ax=axes[0, 1],
    title="Detection+ coverage by delta_k",
    ylabel="Coverage (%)",
)
plot_metric_by_delta_k(
    presentation_results,
    "observation_plus_pct",
    ax=axes[0, 2],
    title="Observation+ coverage by delta_k",
    ylabel="Coverage (%)",
)
plot_metric_by_delta_k(
    presentation_results,
    "recognition_plus_pct",
    ax=axes[1, 0],
    title="Recognition+ coverage by delta_k",
    ylabel="Coverage (%)",
)
plot_metric_by_delta_k(
    presentation_results,
    "identification_pct",
    ax=axes[1, 1],
    title="Identification coverage by delta_k",
    ylabel="Coverage (%)",
)
plot_metric_by_delta_k(
    presentation_results,
    "blind_spot_pct",
    ax=axes[1, 2],
    title="Blind-spot percentage by delta_k",
    ylabel="Open-cell blind spots (%)",
)
plt.tight_layout()
plt.show()


## Representative Spatial Outputs

For each floor plan, the spatial outputs below show three checkpoints:

- the aligned starting line at `K = min_k`
- a middle checkpoint from the available budget sweep
- the upper configured budget at `K = 25`

This makes it easier to see how the camera layout and DORI map evolve as the budget increases.

In [ ]:
figure, axes = plt.subplots(len(presentation_results), 3, figsize=(15, 4 * len(presentation_results)))
if len(presentation_results) == 1:
    axes = [axes]

for row_axes, result in zip(axes, presentation_results):
    min_k = require_floorplan_min_k(result.floorplan)
    showcase_k_values = showcase_k_values_by_floorplan[result.floorplan.name]
    for axis, k_value in zip(row_axes, showcase_k_values):
        artifacts = result.get_visualization_artifacts(k_value)
        plot_dori_map(
            result.floorplan,
            artifacts,
            ax=axis,
            title=f"{result.floorplan.name} DORI @ K={k_value} (delta_k={k_value - min_k})",
        )

plt.tight_layout()
plt.show()

figure, axes = plt.subplots(len(presentation_results), 3, figsize=(15, 4 * len(presentation_results)))
if len(presentation_results) == 1:
    axes = [axes]

for row_axes, result in zip(axes, presentation_results):
    min_k = require_floorplan_min_k(result.floorplan)
    showcase_k_values = showcase_k_values_by_floorplan[result.floorplan.name]
    for axis, k_value in zip(row_axes, showcase_k_values):
        artifacts = result.get_visualization_artifacts(k_value)
        plot_selected_configurations(
            result.floorplan,
            artifacts,
            ax=axis,
            title=f"{result.floorplan.name} cameras @ K={k_value} (delta_k={k_value - min_k})",
        )

plt.tight_layout()
plt.show()


## Closing Interpretation

Across all four floor plans, increasing `K` raises the total DORI score and generally improves the percentage of open cells that reach Detection, Observation, Recognition, and Identification levels, while reducing blind spots. The absolute-`K` view shows how each layout improves under the same raw budget, and the `delta_k` view shows how those improvements compare once each layout begins from its own preferred starting line.

The `min_k` metadata is used **only** to align presentation comparisons. It does not change the optimization model or the model settings used in this analysis.